# HDF5 test

In [43]:
import numpy as np
import pandas as pd
import seaborn as sns
import h5py # for reading hdf5
import os
import fnmatch

# Print all output of a given code chunk instead of just the last line:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Importing an HDF5
The `File` object is the starting point. h5py treats HDF5 files like python dictionaries.

In [27]:
fp = "./data/NEON/NEON_eddy-flux/NEON.D17.SJER.DP4.00200.001.2022-12.basic.20240127T000425Z.RELEASE-2024/NEON.D17.SJER.DP4.00200.001.nsae.2022-12.basic.20240122T083030Z.h5"
f = h5py.File(fp)

We can check the dictionary keys as such:

In [3]:
list(f.keys())

['SJER', 'objDesc', 'readMe']

And we can index the file using the structure defined by NEON (see the file `NEON_how_to_view_hdf5_vA.pdf` for a diagram). Here are the types of data:

In [6]:
f['SJER']['dp01']['data'].keys()

<KeysViewHDF5 ['amrs', 'ch4Conc', 'co2Stor', 'co2Turb', 'fluxHeatSoil', 'h2oSoilVol', 'h2oStor', 'h2oTurb', 'isoCo2', 'isoH2o', 'presBaro', 'radiNet', 'soni', 'tempAirLvl', 'tempAirTop', 'tempSoil']>

Let's look at soil temperature:

In [14]:
f['SJER']['dp01']['data']['ch4Conc']['000_010_09m']['rtioMoleDryCh4']

<HDF5 dataset "rtioMoleDryCh4": shape (766,), type "|V84">

This dataset isn't an array but an HDF5 dataset. Similar to NumPy arrays, they have both a shape and a data type.

In [49]:
ch4_conc = f['SJER']['dp01']['data']['ch4Conc']['000_010_09m']['rtioMoleDryCh4']
ch4_conc.shape
ch4_conc.dtype


(751,)

dtype([('mean', '<f8'), ('min', '<f8'), ('max', '<f8'), ('vari', '<f8'), ('numSamp', '<i4'), ('timeBgn', 'S24'), ('timeEnd', 'S24')])

Convert to a pandas dataframe:

In [35]:
ch4_conc_df = pd.DataFrame(np.array(ch4_conc))

In [ ]:
sns.relplot(
    data = ch4_conc_df,
    x = "mean", y = "timeBgn"
)

# Find all


In [47]:
def find_h5_files(directory):
    h5_files = []
    for root, dirs, files in os.walk(directory):
        for filename in fnmatch.filter(files, '*.h5'):
            h5_files.append(os.path.join(root, filename))
    return h5_files

h5_file_paths = find_h5_files("./data/NEON/")

In [ ]:
aggregated_dataset = []

for fp in h5_file_paths:
    f = h5py.File(fp)
    requested_dataset = f['SJER']['dp01']['data']['ch4Conc']['000_010_09m']['rtioMoleDryCh4']
    df_requested = pd.DataFrame(np.array(requested_dataset))
    aggregated_dataset.append(df_requested)
    
aggregated_dataset

test = pd.DataFrame(aggregated_dataset)

In [55]:
def aggregate_h5_data(h5_files):
    dataframes = []
    k = 0
    n = len(h5_files)
    for file in h5_files:
        try:
            f = h5py.File(fp)
            requested_dataset = f['SJER']['dp01']['data']['ch4Conc']['000_010_09m']['rtioMoleDryCh4']
            df_requested = pd.DataFrame(np.array(requested_dataset))
            dataframes.append(df_requested)
            print("Read file" + k + "of" + n)
        except Exception as e:
            print(f"Could not read {file}: {e}")
    if dataframes:
        return pd.concat(dataframes, ignore_index=True)
    else:
        return pd.DataFrame()  
    
aggregate_h5_data(h5_file_paths)

Could not read ./data/NEON/NEON_eddy-flux/NEON.D17.SJER.DP4.00200.001.2024-08.basic.20240910T191203Z.PROVISIONAL/NEON.D17.SJER.DP4.00200.001.nsae.2024-08.basic.20240909T220550Z.h5: can only concatenate str (not "int") to str
Could not read ./data/NEON/NEON_eddy-flux/NEON.D17.SJER.DP4.00200.001.2023-01.basic.20240127T000425Z.RELEASE-2024/NEON.D17.SJER.DP4.00200.001.nsae.2023-01.basic.20240122T084259Z.h5: can only concatenate str (not "int") to str
Could not read ./data/NEON/NEON_eddy-flux/NEON.D17.SJER.DP4.00200.001.2023-02.basic.20240127T000425Z.RELEASE-2024/NEON.D17.SJER.DP4.00200.001.nsae.2023-02.basic.20240122T085807Z.h5: can only concatenate str (not "int") to str
Could not read ./data/NEON/NEON_eddy-flux/NEON.D17.SJER.DP4.00200.001.2023-11.basic.20240224T203126Z.PROVISIONAL/NEON.D17.SJER.DP4.00200.001.nsae.2023-11.basic.20240223T210605Z.h5: can only concatenate str (not "int") to str
Could not read ./data/NEON/NEON_eddy-flux/NEON.D17.SJER.DP4.00200.001.2024-07.basic.20240808T20170

,mean,min,max,vari,numSamp,timeBgn,timeEnd
0,1.985769,1.982,1.992,5.435896e-06,52,b'2023-05-01T00:28:16.000Z',b'2023-05-01T00:37:15.000Z'
1,2.001055,1.998,2.004,3.534049e-06,55,b'2023-05-01T01:18:16.000Z',b'2023-05-01T01:27:15.000Z'
2,2.049277,2.044,2.055,9.508857e-06,47,b'2023-05-01T02:08:16.000Z',b'2023-05-01T02:17:15.000Z'
3,2.056109,2.055,2.057,3.656463e-07,46,b'2023-05-01T02:58:16.000Z',b'2023-05-01T03:07:15.000Z'
4,2.059040,2.058,2.060,2.840387e-07,50,b'2023-05-01T03:48:16.000Z',b'2023-05-01T03:57:15.000Z'
...,...,...,...,...,...,...,...
15766,2.041055,2.037,2.042,1.200724e-06,55,b'2023-05-31T19:07:40.000Z',b'2023-05-31T19:16:39.000Z'
15767,2.031641,2.030,2.034,6.575364e-07,53,b'2023-05-31T20:07:40.000Z',b'2023-05-31T20:16:39.000Z'
15768,2.027000,2.025,2.029,7.857557e-07,57,b'2023-05-31T21:07:40.000Z',b'2023-05-31T21:16:39.000Z'
15769,2.032283,2.028,2.035,7.860576e-06,53,b'2023-05-31T22:07:40.000Z',b'2023-05-31T22:16:39.000Z'
